# Reproducing Bouabid, Vu et al. (2026) from dandiset 001692

**Paper.** Bouabid S, Vu M-AT, Noggle C, Vietti-Michelina S, Brimblecombe K, Platt N, Zhang L,
Joshi A, Cragg S, Howe MW (2026). An anatomical hotspot for striatal dopamine-acetylcholine
interactions during reward and movement. bioRxiv.
[doi:10.64898/2026.01.20.700614](https://doi.org/10.64898/2026.01.20.700614). This attempt used the
full text of the preprint (version 1, CC-BY).

**Dandiset.** [DANDI:001692](https://dandiarchive.org/dandiset/001692/0.260303.1724), version
0.260303.1724. The paper's Key Resources Table (Zenodo
[doi:10.5281/zenodo.18202873](https://doi.org/10.5281/zenodo.18202873)) and its code's README cite
this version, which is also the only published one, so it is pinned here.

**What the dandiset holds.** Multi-fiber photometry of DA (GRAB-rDA3m, red) and ACh (GRAB-ACh3.0,
green) in head-fixed mice: spontaneous and unpredicted-reward sessions of 5 mice (UG27 to UG31,
Figures 1 to 3, 5 and 6), Pavlovian sessions of 3 mice (AD1 to AD3, Figure 4), sessions of 3 control
mice with non-functional sensors (AD4 to AD6, Extended Data Figure 1), and stimulation sessions of 6
DAT-cre mice (ADS*, Figure 7). The ex vivo slice data of Figure 8 are not in it; their source data
are on Zenodo and are used for the Figure 8 claims.

**Code read.** The paper's analysis code, github.com/HoweLab/BouabidVu2026 at commit 833fa3d
(MATLAB). Where the Methods leave a choice open the code's choice is followed; where they disagree
the Methods are followed and the code's choice is a variant.

| Claim | Tested here |
|---|---|
| fig1e-dominant-polarity: dominant correlation signs and lead (132/127, 42/31 of 174) | yes |
| fig1f-adls-hotspot: negative hotspot in aDLS | yes |
| fig1h-in-vs-out: more negative inside the hotspot | yes |
| fig1f-positive-posterior: positive correlations posterior | yes |
| ed2-hotspot-per-mouse: in-out difference in each mouse | yes |
| ed1-controls-no-organization: no hotspot in control mice | yes |
| fig2a-directionality: DA leads opposite-sign pairs, ACh same-sign pairs | yes |
| fig2-magnitude-correlations: 121, 81, 97, 78 of 174 sites | yes |
| fig2d-magnitude-in-out: opposite-sign magnitude correlations stronger inside | yes |
| fig2b-pairing-overlaps-hotspot: pairing hotspots overlap the correlation hotspot | yes |
| ed3e-transient-rates-in-hotspot: higher transient rates inside | yes |
| fig3b-reward-correlation: 114 negative, 58 positive, latencies | yes |
| fig3d-reward-in-out: more negative inside | yes |
| fig3e-reward-hotspot-overlap: reward hotspot overlaps | yes |
| fig4-cue-sequence: ACh peak, DA peak, ACh dip at 49 of 75 sites | yes |
| fig4c-cue-correlation: 39 negative, 26 positive, latencies | yes |
| fig4h-extinction-correlation: 41 of 74, lag 117.9 ms | yes |
| fig4d-cue-in-out, fig4j-extinction-in-out | yes |
| fig5d-initiation-responses: 131, 119, 73, 32 of 174 | yes |
| fig5-latency-order: DA dip, ACh and DA peaks, ACh dip | yes |
| fig5e-initiation-topography | yes |
| fig6a-initiation-correlation, fig6c-initiation-in-out, fig6g-invigoration-in-out | yes |
| fig7-da-release (179/180), fig7e-ach-suppression (116/180) | yes |
| ed9-cohort-hotspot, fig7f-ach-dip-in-out | yes |
| fig8b, fig8e, fig8fg: slice ANOVAs (Zenodo source data) | yes, data elsewhere |
| fig8c-cocaine-da-similar | blocked (methods underspecified) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

import daach

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
started = time.time()

assets = daach.list_assets()
subject_of = lambda a: a["path"].split("/")[0][4:]
UG = [a for a in assets if subject_of(a).startswith("UG")]
ADS = [a for a in assets if subject_of(a).startswith("ADS")]
AD = [a for a in assets if subject_of(a).startswith("AD") and not subject_of(a).startswith("ADS")]

ug = daach.load_many(UG, series="hp_art")
ad_events = daach.load_many(AD, func=daach.read_events)
ads = daach.load_many(ADS, series="raw")


def light_reward_fraction(e):
    ev = e["events"]
    if "Light" not in ev:
        return np.nan
    rewards = ev["Reward"][0] if "Reward" in ev else np.array([])
    return np.mean([((rewards > a) & (rewards < b)).any() for a, b in zip(*ev["Light"])])


pav_table = pd.DataFrame([{"path": e["path"], "mouse": e["subject"], "session": e["session_id"],
                           "indicators": "+".join(e["indicators"]), "events": ",".join(sorted(e["events"])),
                           "n_light": len(e["events"].get("Light", ([],))[0]),
                           "n_reward": len(e["events"].get("Reward", ([],))[0]),
                           "light_rewarded": light_reward_fraction(e)} for e in ad_events])
by_path = {a["path"]: a for a in assets}
selected = []
for m in ("AD1", "AD2", "AD3"):
    t = pav_table[pav_table.mouse == m].sort_values("session")
    selected += [(p, "learning") for p in t[t.light_rewarded == 1.0].path.iloc[-3:]]
    selected += [(p, "extinction") for p in t[np.isclose(t.light_rewarded, 0.2)].path.iloc[-3:]]
pav = daach.load_many([by_path[p] for p, _ in selected], series="hp")
for s, (_, phase) in zip(pav, selected):
    s["phase"] = phase
control_paths = pav_table[pav_table.mouse.isin(["AD4", "AD5", "AD6"]) & (pav_table.n_light == 0)
                          & (pav_table.n_reward > 0)].path
ctrl = daach.load_many([by_path[p] for p in control_paths], series="hp")
print(f"{len(assets)} assets in dandiset {daach.DANDISET_ID} version {daach.DANDISET_VERSION}; streamed "
      f"{len(ug)} UG, {len(ad_events)} AD event tables, {len(pav)} Pavlovian, {len(ctrl)} control and "
      f"{len(ads)} ADS sessions in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
files = pd.DataFrame([{"mouse": subject_of(a), "size_GB": a["size"] / 1e9} for a in assets])
summary = files.groupby("mouse").agg(files=("size_GB", "size"), total_GB=("size_GB", "sum")).round(1)
ind = {s["subject"]: "+".join(s["ACh_indicator"] + s["DA_indicator"]) for s in ug + ads if "missing" not in s}
ind.update({e["subject"]: "+".join(e["indicators"]) for e in ad_events})
summary["indicators"] = summary.index.map(ind)
print(summary.to_string())
print("Subjects:", len(summary), "; the Methods list 17 in vivo mice (5 spontaneous, 3 Pavlovian, 3 control, 6 DAT-cre).")

ug_ok = [s for s in ug if "missing" not in s]
print("\nSessions skipped because their high-pass, artifact-removed series is absent (the code skips empty ones):",
      [(s["path"], s["missing"]) for s in ug if "missing" in s])
spont = [s for s in ug_ok if "Reward" not in s["events"]]
reward = [s for s in ug_ok if "Reward" in s["events"]]
print(f"UG27-UG31 sessions: {len(ug)} files, {len(spont)} without rewards (spontaneous), {len(reward)} with rewards")
print("Event types in spontaneous sessions:", sorted({k for s in spont for k in s["events"]}))

In [ ]:
desc = daach.load_many([by_path["sub-UG27/sub-UG27_ses-240214r_ophys.nwb"], AD[[subject_of(a) for a in AD].index("AD4")]],
                       func=daach.read_descriptions)
for name, d in zip(("UG27", "AD4"), desc):
    print(f"--- {name}")
    for k in ("ExcitationSource470", "ExcitationSource570", "ACh3.0", "rDA3m", "ACh4.3mut", "tdTomato"):
        if k in d:
            print(f"{k}: {d[k]!r}")
    print("session_description:", repr(d["session_description"][:420]))
    print("pharmacology (control virus):", repr(d["pharmacology"][-260:]))

The files of AD4 to AD6 name the mutant ACh sensor and tdTomato; all others name ACh3.0 and rDA3m.
The files call the mutant sensor ACh4.3mut (device and surgery text), where the Methods and the Key
Resources Table name ACh3.0-mut. The description of the 470 nm source describes a different setup
(470 and 405 nm alternated at 100 Hz, with an isosbestic control), and that of the 570 nm source
reads "405 nm"; the Methods alternate 470 and 570 nm at 36 Hz. The session description gives 9 uL
rewards at 5 to 30 s intervals, where the Methods give 5 uL at 10 to 30 s, and its micro sign is
stored mis-encoded ("9 Î¼L").
The cohorts are told apart only by subject name and indicator; no field in the files states the
experiment or the figure a session belongs to. UG29's session 240308 has licks but no rewards and is
treated as spontaneous, as its name has no `r`.

In [ ]:
rows = []
for name, ss in (("UG", ug_ok), ("Pavlovian", pav), ("control", ctrl), ("ADS", ads)):
    for m, sm_ in daach.by_mouse(ss).items():
        c0 = sm_[0]["coords"]
        same = all(s["coords"].shape == c0.shape and np.allclose(s["coords"], c0) for s in sm_)
        rows.append({"cohort": name, "mouse": m, "sessions": len(sm_), "fibers_per_channel": sm_[0]["ACh_n_fibers"],
                     "included_sites": len(c0), "coords_same_in_all_sessions": same,
                     "AP_range": f"{c0[:, 0].min():.2f} to {c0[:, 0].max():.2f}",
                     "ML_range": f"{c0[:, 1].min():.2f} to {c0[:, 1].max():.2f}",
                     "DV_range": f"{c0[:, 2].min():.2f} to {c0[:, 2].max():.2f}"})
sites = pd.DataFrame(rows)
print(sites.to_string(index=False))
print("\nIncluded sites per cohort:", sites.groupby("cohort").included_sites.sum().to_dict())
print("Paper: 174 sites in 5 mice (Figures 1-3, 5, 6); 75 sites in 3 mice (Figure 4); 180 sites in 6 mice (Figure 7).")

The included sites match the paper's counts: 174 for UG27 to UG31, 75 for AD1 to AD3 and 180 for
the six ADS mice. The fiber coordinates (mm from bregma, AP, ML and DV) are the same in every session
of a mouse, which is consistent with each site being the same fiber across sessions; the analysis
treats it so, as the paper does, and the per-mouse breakdowns below are the check on pooling. All
sites lie in the right hemisphere (ML 0.37 to 3.74 mm).

In [ ]:
rows = []
for s in ug_ok + pav + ctrl + ads:
    t = s["t"]
    rows.append({"mouse": s["subject"], "n": len(s["DA"]), "stated_rate_green": s["ACh_stated_rate"],
                 "stated_rate_red": s["DA_stated_rate"], "behavior_dt_ms": np.median(np.diff(t)) * 1000,
                 "behavior_rate": (len(t) - 1) / (t[-1] - t[0]), "n_behavior": len(t),
                 "green_start_minus_behavior_ms": (s["ACh_start"] - t[0]) * 1000,
                 "red_start_minus_green_ms": (s["DA_start"] - s["ACh_start"]) * 1000,
                 "minutes_at_stated_rate": len(s["DA"]) / s["ACh_stated_rate"] / 60,
                 "minutes_from_timestamps": (t[-1] - t[0]) / 60})
clock = pd.DataFrame(rows)
print(clock.describe().loc[["min", "50%", "max"]].round(3).to_string())
print("\nSessions whose photometry and behavior sample counts differ:", int((clock.n != clock.n_behavior).sum()))

Every photometry series states a rate of 9.0 Hz, but each has as many samples as the behavior
series, starts at the behavior's first timestamp, and the behavior timestamps step by 55.6 ms
(18.0 Hz). The Methods give 18 Hz per channel and the code uses `sr = 18`. At the stated rate the
recordings would last twice as long as their timestamps. The data support 18 Hz; it is used
throughout, and the stated rate is a metadata error. The red series starts 27.5 ms (half a sample)
before the green, as the alternating excitation implies; DA and ACh are paired by sample index, as
in the code.

In [ ]:
rows = []
for name, ss in (("UG hp_art", ug_ok), ("Pavlovian hp", pav), ("control hp", ctrl), ("ADS raw", ads)):
    for ch in ("DA", "ACh"):
        x = np.concatenate([s[ch].ravel() for s in ss])
        flat = sum(int((np.nanstd(s[ch], 0) == 0).sum()) for s in ss)
        rows.append({"series": name, "channel": ch, "p1": np.nanpercentile(x, 1), "median": np.nanmedian(x),
                     "p99": np.nanpercentile(x, 99), "NaN_fraction": np.isnan(x).mean(),
                     "constant_site_sessions": flat})
print("dF/F (dimensionless; the files state the unit 'n.a.'):")
print(pd.DataFrame(rows).round(4).to_string(index=False))
vel = pd.DataFrame([{"mouse": s["subject"], "min": s["velocity"].min(), "median": np.median(s["velocity"]),
                     "p99": np.percentile(s["velocity"], 99), "max": s["velocity"].max(),
                     "frac_above_0.0208": (daach.smooth_velocity(s["velocity"]) > 0.0208).mean()} for s in spont])
print("\nWheel velocity (m/s) in spontaneous sessions, per mouse (median over sessions):")
print(vel.groupby("mouse").median().round(4).to_string())

The dF/F values are small fractions, as dF/F should be (1st to 99th percentiles between -0.033 and 0.040).
The artifact-removed series holds NaNs where artifacts were cut. Some control sites have a constant
red trace in some sessions; their correlation is undefined there and is averaged over the other
sessions. Wheel velocity is in m/s with plausible values (99th percentiles 0.13 to 0.47 m/s). The mice
differ in how much they run: the smoothed velocity exceeds the stillness threshold 11% to 79% of the time.

In [ ]:
pav_table["has_hp_art"] = [any("HpArt" in k for k in e["series"]) for e in ad_events]
print("AD files holding an artifact-removed (HpArt) series:", pav_table.groupby("mouse").has_hp_art.sum().to_dict(),
      "of", pav_table.groupby("mouse").size().to_dict())
print(pav_table.groupby(["mouse", "events", "light_rewarded"], dropna=False).size().rename("sessions").to_string())
print("\nSelected Pavlovian sessions (last 3 per phase, as the code takes):")
print(pd.DataFrame([{"mouse": s["subject"], "session": s["session_id"], "phase": s["phase"],
                     "light_cues": len(s["events"]["Light"][0]), "rewards": len(s["events"]["Reward"][0])}
                    for s in pav]).to_string(index=False))
print("\nControl sessions used:", len(ctrl), {m: len(v) for m, v in daach.by_mouse(ctrl).items()})
stim_counts = pd.DataFrame([{"mouse": s["subject"], "session": s["session_id"],
                             "stimulations": len(s["events"].get("Optogenetic Stimulation", ([],))[0]),
                             "rewards": len(s["events"].get("Reward", ([],))[0])} for s in ads])
print("\nStimulation sessions:")
print(stim_counts.to_string(index=False))

Pavlovian sessions fall into three groups by the fraction of light cues followed by a reward: 1.0
(learning), 0.2 (light cue extinguished) and 0.8 (tone cue extinguished). Learning sessions have 68
rewards: 30 after light cues, 30 after tone cues and 8 uncued, as the Methods describe. The files
hold no tone cue events, so tone trials cannot be analysed; the paper's Figure 4 uses light trials
only, as its code shows. Which phase a session belongs to is not stated in the files and is inferred
here from the events.

In [ ]:
def fingerprint(s):
    return (len(s["DA"]), tuple(np.round(np.nan_to_num(s["DA"][:200, :3]).ravel(), 6)))

seen, dups = {}, []
for s in ug_ok + pav + ctrl + ads:
    k = fingerprint(s)
    if k in seen:
        dups.append((seen[k], s["path"]))
    seen[k] = s["path"]
print("Sessions with identical length and identical first 200 DA samples at 3 sites:", dups if dups else "none")
same_len = pd.Series([len(s["DA"]) for s in ug_ok]).value_counts()
print("UG sessions sharing a sample count with another:", int((same_len > 1).sum()), "counts;",
      "none share traces (above)")

In [ ]:
def response_contrast(sessions, events_of, shift_s=10.0):
    # mean over sites of |mean dF/F 0-1 s minus -0.5-0 s| at the events and at events shifted later
    out = []
    for m, ss in daach.by_mouse(sessions).items():
        vals = {"true": [], "shifted": []}
        for s in ss:
            e = events_of(s)
            for key, ev in (("true", e), ("shifted", e + int(shift_s * daach.SR))):
                ev = ev[(ev > 9) & (ev < len(s["DA"]) - 19)]
                if not len(ev):
                    continue
                for ch in ("DA", "ACh"):
                    r = daach.raster(s[ch], ev, -9, 18)
                    mu = np.nanmean(r, 2)
                    vals[key].append(np.nanmean(np.abs(mu[9:].mean(0) - mu[:9].mean(0))))
        out.append({"mouse": m, "true": np.mean(vals["true"]), "shifted_10s": np.mean(vals["shifted"])})
    return pd.DataFrame(out)

checks = {
    "reward sessions, first lick": (reward, lambda s: daach.first_licks(s)[1]),
    "Pavlovian, light cue": (pav, lambda s: daach.event_index(s, s["events"]["Light"][0])),
    "ADS, stimulation": (ads, lambda s: daach.event_index(s, s["events"]["Optogenetic Stimulation"][0])),
    "spontaneous, initiation": (spont, lambda s: daach.initiations(daach.smooth_velocity(s["velocity"]))[0]),
}
for name, (ss, f) in checks.items():
    t = response_contrast(ss, f)
    t["ratio"] = t.true / t.shifted_10s
    print(name)
    print(t.round(4).to_string(index=False))

For every mouse and each kind of event, the photometry response in the second after the event is
larger at the event times than at times shifted by 10 s (the ratios above), so the events, the
velocity and the photometry share one clock in every cohort. This shows alignment to within the
response's time scale (about a second); finer offsets would not show here.

## fig1e-dominant-polarity

> Across 174 sites in 5 mice, the dominant DA-ACh cross-correlation during spontaneous behavior is
> negative at 132 sites, 127 of them with DA leading, and positive at 42 sites, 31 of them with ACh
> leading.

In [ ]:
xc1 = daach.spontaneous_xcorr(spont)
mice1, coords1 = daach.site_table(spont)
dom1, lag1 = xc1["dominant"], xc1["lag"]


def polarity_counts(dom, lag):
    return {"negative": int((dom < 0).sum()), "negative, DA leads": int(((dom < 0) & (lag > 0)).sum()),
            "positive": int((dom > 0).sum()), "positive, ACh leads": int(((dom > 0) & (lag < 0)).sum()),
            "lag 0": int((lag == 0).sum()), "sites": len(dom)}

expected = {"negative": 132, "negative, DA leads": 127, "positive": 42, "positive, ACh leads": 31}
c1 = polarity_counts(dom1, lag1)
print("Counts:", c1)
print("Within 10% of the paper:", {k: abs(c1[k] - v) <= 0.1 * v for k, v in expected.items()})
per = pd.DataFrame([{"mouse": m, **polarity_counts(dom1[mice1 == m], lag1[mice1 == m])} for m in np.unique(mice1)])
print(per.to_string(index=False))
print("Dominant r range", dom1.min().round(3), "to", dom1.max().round(3), "; latency range (ms)",
      lag1.min() * 1000 / 18, "to", lag1.max() * 1000 / 18)
xc1c = daach.spontaneous_xcorr(spont, concatenate=True)
print("Variant, correlation over concatenated sessions:", polarity_counts(xc1c["dominant"], xc1c["lag"]))

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.5))
ax.scatter(lag1 * 1000 / 18, dom1, s=10, c=np.where(dom1 < 0, "tab:blue", "tab:red"))
ax.axhline(0, color="k", lw=0.5)
ax.axvline(0, color="k", lw=0.5)
ax.set_xlabel("latency (ms), ACh leads <- -> DA leads")
ax.set_ylabel("dominant Pearson r")
ax.set_title("Spontaneous DA-ACh, 174 sites")
plt.show()
print("All points are inside the axes (latency -1000 to 1000 ms).")

**Result: reproduced.** Of 174 sites, the dominant correlation is negative at 133 (128 with DA
leading) and positive at 41 (32 with ACh leading), against 132, 127, 42 and 31 in the paper; all
four are within 10%. Every mouse has more negative than positive sites. Correlating concatenated
sessions gives the same counts except 30 positive sites with ACh leading.

## fig1f-adls-hotspot

> The dominant correlations form a significant hotspot, identified by local Moran's I, in the
> anterior dorsolateral striatum (aDLS), where the strongest correlations are negative.

The hotspot procedure is fixed in the claims file: linear interpolation onto a 100 um grid, padding,
Gaussian smoothing, local Moran's I against 200 shuffles of the values across sites, the 95th
percentile, and the connected volume of at least 0.125 mm3 with the largest mean magnitude. One grid
is shared by all cohorts so that sites of other cohorts can be placed in this hotspot.

In [ ]:
mice_c, coords_c = daach.site_table(ctrl)
mice_p, coords_p = daach.site_table(pav)
mice_s, coords_s = daach.site_table(ads)
GRID = daach.Grid(np.concatenate([coords1, coords_c, coords_p, coords_s]))
interp1 = daach.Interpolator(GRID, coords1)
rng = np.random.default_rng(101)
hs1 = daach.hotspot(interp1, dom1, rng)
in1 = daach.inside(GRID, hs1, coords1)
med = np.median(coords1, 0)
print("Grid (AP, ML, DV voxels):", GRID.shape, "; voxels inside the convex hull of the sites:", int(interp1.inside.sum()))
print("Candidate components:", hs1["n_components"], "; hotspot voxels:", len(hs1["voxels"]),
      f"({len(hs1['voxels']) * 1e-3:.3f} mm3)")
print(f"Hotspot mean smoothed r {hs1['mean']:.3f}; striatal (hull) mean {hs1['striatal_mean']:.3f}")
print("Centroid (AP, ML, DV mm):", hs1["centroid"].round(2), "; median site coordinates:", med.round(2))
print("Anterior, lateral, dorsal of the median:", list(hs1["centroid"] > med))
print("Sites inside:", int(in1.sum()), "by mouse:", pd.Series(mice1[in1]).value_counts().to_dict())
for pct, min_vox in ((99, 125), (95, 1000), (90, 125)):
    v = daach.hotspot(interp1, dom1, rng, pct=pct, min_vox=min_vox, null=hs1["null"])
    print(f"Variant, null percentile {pct}, minimum {min_vox * 1e-3:.3f} mm3: components {v['n_components']}, "
          f"mean {v['mean']:.3f}, voxels {0 if v['voxels'] is None else len(v['voxels'])}, centroid {v['centroid'].round(2)}")

In [ ]:
def show_map(hs, title, vmax=0.5):
    vol = hs["smooth"]
    mask = np.zeros(GRID.shape, bool)
    if hs["voxels"] is not None:
        mask.ravel()[hs["voxels"]] = True
    fig, axs = plt.subplots(1, 2, figsize=(8, 3.5))
    ap, ml, dv = GRID.axes
    axs[0].imshow(np.nanmean(vol, 2), origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto",
                  extent=[ml[0], ml[-1], ap[0], ap[-1]])
    axs[0].contour(ml, ap, mask.any(2), levels=[0.5], colors="k")
    axs[0].set_xlabel("ML (mm)"); axs[0].set_ylabel("AP (mm)"); axs[0].set_title("axial, mean projection")
    im = axs[1].imshow(np.nanmean(vol, 1).T, origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto",
                       extent=[ap[0], ap[-1], dv[0], dv[-1]])
    axs[1].contour(ap, dv, mask.any(1).T, levels=[0.5], colors="k")
    axs[1].set_xlabel("AP (mm)"); axs[1].set_ylabel("DV (mm)"); axs[1].set_title("sagittal, mean projection")
    axs[1].invert_xaxis()
    fig.colorbar(im, ax=axs, shrink=0.8)
    fig.suptitle(title)
    plt.show()
    print(f"Smoothed values range {np.nanmin(vol):.3f} to {np.nanmax(vol):.3f}; color scale clipped at +/-{vmax}; "
          f"voxels beyond it: {int((np.abs(vol) > vmax).sum())} of {int((~np.isnan(vol)).sum())}")

show_map(hs1, "Dominant spontaneous DA-ACh r; black, hotspot")

**Result: partially reproduced.** A significant hotspot is found (1.01 mm3) with a negative mean
smoothed r (-0.368, against -0.185 over the hull). Its centroid (AP 0.63, ML 2.02, DV -3.14 mm) is
anterior and dorsal of the median site (0.10, 2.04, -3.21) but 0.02 mm medial of it, so by the
criterion fixed beforehand it is not in the lateral half. With the 99th percentile of the null the
hotspot shrinks to 518 voxels and its centroid moves to ML 2.09 mm, lateral of the median. Sites of
all five mice fall inside it (28 sites).

## fig1h-in-vs-out

> Sites inside the aDLS hotspot have more negative dominant correlations than sites outside
> (Wilcoxon rank-sum p = 1.67e-16).

In [ ]:
r = daach.in_out(dom1, in1)
print(r)
print("Paper: p = 1.67e-16. The test is partly circular: the hotspot is drawn from the same values.")

**Result: reproduced.** The 28 sites inside the hotspot have a median dominant r of -0.41
against -0.14 for the 146 outside (rank-sum p = 1.8e-13; the paper 1.67e-16).

## fig1f-positive-posterior

> Dominant positive correlations lie in the posterior striatum.

In [ ]:
ap = coords1[:, 0]
t = stats.mannwhitneyu(ap[dom1 > 0], ap[dom1 < 0], alternative="less")
print(f"AP of positive sites: median {np.median(ap[dom1 > 0]):.2f} mm (n {int((dom1 > 0).sum())}); "
      f"negative sites: median {np.median(ap[dom1 < 0]):.2f} mm (n {int((dom1 < 0).sum())}); one-sided p = {t.pvalue:.3g}")
for m in np.unique(mice1):
    k = mice1 == m
    a, b = ap[k & (dom1 > 0)], ap[k & (dom1 < 0)]
    p = stats.mannwhitneyu(a, b, alternative="less").pvalue if len(a) and len(b) else np.nan
    print(f"  {m}: positive n {len(a)}, median AP {np.median(a) if len(a) else np.nan:.2f}; negative n {len(b)}, "
          f"median AP {np.median(b):.2f}; p = {p:.3g}")
print("Variant, positive sites against all others (DV and ML medians too):",
      {"ML_pos": np.median(coords1[dom1 > 0, 1]).round(2), "ML_neg": np.median(coords1[dom1 < 0, 1]).round(2),
       "DV_pos": np.median(coords1[dom1 > 0, 2]).round(2), "DV_neg": np.median(coords1[dom1 < 0, 2]).round(2)})

**Result: reproduced.** Sites with a positive dominant correlation lie more posterior (median AP
-0.40 mm, 41 sites) than sites with a negative one (0.24 mm, 133 sites), one-sided p = 4.3e-07. All
five mice show the same direction, significant in three (UG27, UG29, UG31).

## ed2-hotspot-per-mouse

> The hotspot is consistent across mice; in each mouse, sites inside the group hotspot have more
> negative dominant correlations than sites outside.

In [ ]:
rows = []
for m in np.unique(mice1):
    k = mice1 == m
    rows.append({"mouse": m, **daach.in_out(dom1[k], in1[k])})
per2 = pd.DataFrame(rows)
print(per2.round(4).to_string(index=False))
ok = per2[(per2.n_in >= 2) & (per2.n_out >= 2)]
print("Mice tested (>= 2 sites each side):", len(ok), "; median inside lower:", int((ok.median_in < ok.median_out).sum()),
      "; and p < 0.05:", int(((ok.median_in < ok.median_out) & (ok.p < 0.05)).sum()))

**Result: reproduced.** In all five mice the median dominant r is lower inside the group hotspot
than outside, with p < 0.05 in four (UG27, UG28, UG30, UG31); UG29 has 2 sites inside (p = 0.079).

## ed1-controls-no-organization

> Mice expressing non-functional sensors (n = 3) show only sparse, weak positive red-green
> correlations with no anatomical organization; no hotspot is found.

In [ ]:
xcc = daach.spontaneous_xcorr(ctrl, mask=daach.reward_mask)
domc = xcc["dominant"]
keep = ~np.isnan(domc)
print("Control sites:", len(domc), "; with no defined correlation in any session (dropped):", int((~keep).sum()))
domc, mice_ck, coords_ck = domc[keep], mice_c[keep], coords_c[keep]
print(f"Control dominant r: median {np.median(domc):.3f}, positive at {(domc > 0).mean():.2f} of sites, "
      f"median |r| {np.median(np.abs(domc)):.3f}; sensor mice median |r| {np.median(np.abs(dom1)):.3f}")
print(f"|r| > 0.2: control {(np.abs(domc) > 0.2).mean():.2f}, sensor {(np.abs(dom1) > 0.2).mean():.2f}")
print(pd.DataFrame([{"mouse": m, "sites": int((mice_ck == m).sum()), "median_r": np.median(domc[mice_ck == m]),
                     "frac_positive": (domc[mice_ck == m] > 0).mean()} for m in np.unique(mice_ck)]).round(3).to_string(index=False))
interp_c = daach.Interpolator(GRID, coords_ck)
hsc = daach.hotspot(interp_c, domc, np.random.default_rng(102))
print(f"Hotspot procedure: components {hsc['n_components']}, hotspot mean {hsc['mean']:.3f}, "
      f"voxels {0 if hsc['voxels'] is None else len(hsc['voxels'])}, centroid {hsc['centroid'].round(2)}")
xcc2 = daach.spontaneous_xcorr(ctrl)
d2 = xcc2["dominant"][~np.isnan(xcc2["dominant"])]
print(f"Variant, reward periods kept: median r {np.median(d2):.3f}, positive {(d2 > 0).mean():.2f}")

**Result: not reproduced.** In the control mice the hotspot procedure finds a hotspot (mean
smoothed r +0.33, 551 voxels, centroid AP 0.68, ML 1.29, DV -4.14 mm). Their correlations are mostly
positive (68% of 124 sites, median r 0.20) but not weak: the median |r| is 0.238, as in the sensor
mice (0.240), and 65% of control sites exceed |r| = 0.2 against 54% of sensor sites. The test used
the 15 uncued reward sessions of these mice, not all of their ITI data.

## fig2a-directionality

> For coincident transients of opposite polarity the DA transient leads; for same-polarity pairs the
> ACh transient leads.

In [ ]:
t0 = time.time()
tp = pd.DataFrame(daach.transient_pairs(ug_ok, np.random.default_rng(201)))
print(f"Transients and pairs computed over {len(ug_ok)} sessions in {time.time() - t0:.0f} s; sites {len(tp)}")
names = [daach.pair_name(t) for t in daach.PAIR_TYPES]
rows = []
for k in names:
    di = tp[f"di|{k}"].dropna()
    rows.append({"pair": k, "sites": len(di), "median_DI": di.median(), "frac_negative": (di < 0).mean(),
                 "wilcoxon_p": stats.wilcoxon(di).pvalue})
print(pd.DataFrame(rows).to_string(index=False))
print("\nMedian directionality index by mouse (negative: DA leads):")
print(tp.groupby("mouse")[[f"di|{k}" for k in names]].median().round(3).to_string())

**Result: reproduced.** The median directionality index is -0.27 (ACh trough with DA peak) and
-0.30 (ACh peak with DA trough), DA leading, and +0.33 (peak with peak) and +0.12 (trough with
trough), ACh leading, each with Wilcoxon p below 1e-24 over 174 sites. Each mouse has the same four
signs.

## fig2-magnitude-correlations

> Magnitudes of paired transients are negatively correlated at many sites for opposite-sign pairs
> (121 and 81 of 174) and positively for same-sign pairs (97 and 78 of 174), with no site
> significant in the other direction.

In [ ]:
exp = {names[0]: ("negative", 121), names[1]: ("negative", 81), names[2]: ("positive", 97), names[3]: ("positive", 78)}
rows = []
for k in names:
    neg, pos = int((tp[f"hi|{k}"] < 0).sum()), int((tp[f"lo|{k}"] > 0).sum())
    sign, e = exp[k]
    got = neg if sign == "negative" else pos
    code_neg = int(((tp[f"code_p|{k}"] < 0.05) & (tp[f"code_r|{k}"] < 0)).sum())
    code_pos = int(((tp[f"code_p|{k}"] < 0.05) & (tp[f"code_r|{k}"] > 0)).sum())
    rows.append({"pair": k, "sig_negative": neg, "sig_positive": pos, "expected": f"{e} {sign}",
                 "within_10pct": abs(got - e) <= 0.1 * e, "sites_<100_pairs": int((tp[f"npairs|{k}"] < 100).sum()),
                 "median_pairs": int(tp[f"npairs|{k}"].median()), "median_mean_r": tp[f"mag|{k}"].median(),
                 "code_variant_neg": code_neg, "code_variant_pos": code_pos})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nSignificant sites of the stated sign, by mouse:")
by = {}
for k in names:
    col = (tp[f"hi|{k}"] < 0) if exp[k][0] == "negative" else (tp[f"lo|{k}"] > 0)
    by[k] = col.groupby(tp.mouse).sum()
print(pd.DataFrame(by).to_string())

**Result: partially reproduced.** Significant sites of the stated sign are 115 (ACh trough with
DA peak), 70 (ACh peak with DA trough), 90 (peak with peak) and 76 (trough with trough), against
121, 81, 97 and 78; three are within 10%, and the second is 2.9 below its lower bound (72.9). No site
is significant in the other direction for any type, and every site has at least 100 pairs. The
code's computation, Pearson r over all pairs of the leading order, finds 143 to 150 sites per type.

## fig2d-magnitude-in-out

> Magnitude correlations of opposite-sign pairs, but not same-sign pairs, are stronger inside the
> aDLS hotspot than outside.

In [ ]:
in_tp = in1  # transient_pairs returns sites in the same order as site_table
rows = []
for k in names:
    v = np.where(tp[f"npairs|{k}"] >= 100, tp[f"mag|{k}"], np.nan)
    rows.append({"pair": k, **daach.in_out(v, in_tp)})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("Paper: p = 0.017 and 0.006 for the opposite-sign pairs, 0.091 and 0.612 for the same-sign pairs.")

**Result: partially reproduced.** For ACh troughs with DA peaks the magnitude correlation is more
negative inside the hotspot (median -0.32 against -0.23, p = 0.0001); for ACh peaks with DA troughs
the difference is small and not significant (-0.18 against -0.17, p = 0.12). Neither same-sign type
differs (p = 0.12 and 0.21), as the paper states.

## fig2b-pairing-overlaps-hotspot

> The frequency of paired transients is highest in aDLS, in a region overlapping the correlation
> hotspot, for all four pair types (volume overlap test, p < 0.01 for each).

In [ ]:
rng = np.random.default_rng(202)
rows = []
for k in names:
    occ = tp[f"occ|{k}"].to_numpy()
    h = daach.hotspot(interp1, occ, rng, how="max")
    ov = daach.overlap_test(hs1, interp1.inside, h, interp1.inside, GRID.shape, rng)
    io = daach.in_out(occ, in1)
    rows.append({"pair": k, "median_occurrence": np.median(occ), "hotspot_mean": h["mean"],
                 "striatal_mean": h["striatal_mean"], "hotspot_voxels": ov["n_b"], "overlap_voxels": ov["overlap"],
                 "null_median_overlap": ov.get("null_median"), "overlap_p": ov["p"],
                 "centroid_AP_ML_DV": h["centroid"].round(2), "in_median": io["median_in"], "out_median": io["median_out"],
                 "in_out_p": io["p"]})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("Paper overlap p: 0.005, 0.005, 0.002, < 0.001 (10000 random volumes; here 200, so the smallest possible p is 0.005).")

**Result: reproduced.** For each pair type the occurrence hotspot shares 782 to 829 voxels with
the correlation hotspot, against null medians of 53 to 128, with overlap p of 0.015, 0.020, 0.005 and
below 0.005 (the paper 0.005, 0.005, 0.002 and below 0.001). The occurrence hotspots have centroids
at AP 0.59 to 0.64, ML 1.85 to 2.06 and DV -3.13 to -3.16 mm, and occurrence is higher inside the
correlation hotspot than outside for each type (rank-sum p < 0.001).

## ed3e-transient-rates-in-hotspot

> DA and ACh transient rates are higher at sites inside the global correlation hotspot than outside.

In [ ]:
rows = []
for k in ("DA_pos", "DA_neg", "ACh_pos", "ACh_neg"):
    rows.append({"transients": k, **daach.in_out(tp[f"rate_{k}"].to_numpy(), in1)})
print("Transients per second (reward periods left out):")
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: reproduced.** All four transient types are more frequent inside the hotspot: 0.198
against 0.156 per second for DA peaks, 0.252 against 0.207 for DA troughs, 0.214 against 0.181 for
ACh peaks and 0.236 against 0.196 for ACh troughs (medians; p of 0.003 or less).

## fig3b-reward-correlation

> After unpredicted rewards, trial-by-trial DA peak magnitude correlates with ACh; dominant
> correlations are negative at 114 of 174 sites with DA leading (177.9 +/- 96.7 ms) and positive at
> 58 of 174 with ACh leading (-41.2 +/- 218.9 ms).

In [ ]:
IDX3 = np.arange(-9, 28)
er3 = daach.event_rasters(reward, lambda s: daach.first_licks(s)[1], -9, 27)
res3 = {m: daach.event_correlation(d["rasters"]["DA"], d["rasters"]["ACh"], 1, IDX3, np.arange(0, 19)) for m, d in er3.items()}
dom3 = daach.stack(res3, lambda v: v["dominant"])
lat3 = daach.stack(res3, lambda v: v["lag"]) * 1000 / 18
mice3, coords3 = daach.site_table(reward)
assert np.allclose(coords3, coords1)
print("Rewards with a consumption lick, per mouse:", {m: len(d["events"]) for m, d in er3.items()})


def corr_summary(dom, lat):
    neg, pos = dom < 0, dom > 0
    return {"sites": int((~np.isnan(dom)).sum()), "negative": int(neg.sum()), "negative_lag>0": int((neg & (lat > 0)).sum()),
            "mean_latency_negative_ms": lat[neg].mean(), "sd_negative": lat[neg].std(ddof=1),
            "positive": int(pos.sum()), "positive_lag<0": int((pos & (lat < 0)).sum()),
            "mean_latency_positive_ms": lat[pos].mean(), "sd_positive": lat[pos].std(ddof=1)}

s3 = corr_summary(dom3, lat3)
print({k: round(v, 1) if isinstance(v, float) else v for k, v in s3.items()})
checks = [("negative", 114, 11.4), ("positive", 58, 5.8), ("mean_latency_negative_ms", 177.9, 9.1),
          ("mean_latency_positive_ms", -41.2, 28.7)]
print("Within tolerance:", {k: abs(s3[k] - e) <= t for k, e, t in checks})
print(pd.DataFrame([{"mouse": m, **{k: (round(v, 1) if isinstance(v, float) else v) for k, v in
                     corr_summary(dom3[mice3 == m], lat3[mice3 == m]).items()}} for m in np.unique(mice3)]).to_string(index=False))

**Result: partially reproduced.** The dominant reward correlation is negative at 122 sites with a
mean latency of 172.1 ms (the paper 114 and 177.9 ms) and positive at 52 sites with a mean latency
of -59.8 ms (the paper 58 and -41.2 ms). Three of the four values are within tolerance; the positive
count is 0.2 below its lower bound (52.2). DA leads at 120 of the 122 negative sites and ACh at 37 of
the 52 positive ones. Negative sites outnumber positive ones in each mouse.

## fig3d-reward-in-out

> Reward-evoked DA-ACh correlations are significantly more negative inside the global hotspot than
> outside.

In [ ]:
print(daach.in_out(dom3, in1))
print(pd.DataFrame([{"mouse": m, **daach.in_out(dom3[mice3 == m], in1[mice3 == m])} for m in np.unique(mice3)]).round(4).to_string(index=False))

**Result: reproduced.** Inside the hotspot the median reward r is -0.52 against -0.31 outside
(p = 7.2e-09). The difference has the same sign in every mouse and is significant in three.

## fig3e-reward-hotspot-overlap

> Reward anti-correlations form an aDLS hotspot that significantly overlaps the hotspot of
> spontaneous correlations (volume overlap test, p = 0.008).

In [ ]:
rng = np.random.default_rng(301)
hs3 = daach.hotspot(interp1, dom3, rng)
ov3 = daach.overlap_test(hs1, interp1.inside, hs3, interp1.inside, GRID.shape, rng)
print(f"Reward hotspot: components {hs3['n_components']}, mean {hs3['mean']:.3f}, voxels {ov3['n_b']}, "
      f"centroid {hs3['centroid'].round(2)}")
print("Overlap with the spontaneous hotspot:", ov3)
show_map(hs3, "Dominant reward DA-ACh r; black, reward hotspot", vmax=0.6)

**Result: reproduced.** The reward correlations form a negative hotspot (mean -0.53, 1022
voxels, centroid AP 0.54, ML 2.07, DV -3.26 mm) that shares 721 voxels with the spontaneous hotspot,
against a null median of 116 (overlap p = 0.01; the paper 0.008).

## fig4-cue-sequence

> After Pavlovian conditioning, the conditioned light cue evokes an ACh peak, then a DA peak, then an
> ACh dip at a majority of sites (49 of 75).

In [ ]:
W4 = (-9, 54)
IDX4 = np.arange(W4[0], W4[1] + 1)


def light_events(s):
    return daach.event_index(s, s["events"]["Light"][0])


def iti_exclusion(s):
    cues = light_events(s)
    rew = daach.event_index(s, s["events"]["Reward"][0])
    return [(i, i + int(9 * daach.SR)) for i in cues] + [(i, i + int(5 * daach.SR)) for i in rew]


learn = [s for s in pav if s["phase"] == "learning"]
ext = [s for s in pav if s["phase"] == "extinction"]
rng = np.random.default_rng(401)
er4 = daach.event_rasters(learn, light_events, *W4, exclude_of=iti_exclusion)
sig4 = {w: daach.triggered_significance(er4, *W4, w, rng) for w in ((9, 27), (9, 36))}
mice4, coords4 = daach.site_table(learn)


def sequence(sig, rows, alpha="methods"):
    out = []
    for m in sorted(sig):
        a, d = sig[m]["ACh"], sig[m]["DA"]
        ap, an = a[alpha]
        dp, _ = d[alpha]
        w = slice(rows[0], rows[1] + 1)
        ta, td, tn = np.argmax(a["mean"][w], 0), np.argmax(d["mean"][w], 0), np.argmin(a["mean"][w], 0)
        out.append(ap & dp & an & (ta < td) & (td < tn))
    return np.concatenate(out)

for label, rows in (("(a) 0 to 1 s", (9, 27)), ("(b) 0 to 1.5 s", (9, 36))):
    seq = sequence(sig4[rows], rows)
    seq_code = sequence(sig4[rows], rows, "code")
    print(f"Reading {label}: sequence at {int(seq.sum())} of {len(seq)} sites; by mouse "
          f"{pd.Series(seq).groupby(mice4).sum().to_dict()}; code percentiles {int(seq_code.sum())}")
s = sig4[(9, 27)]
print("Significant features in 0 to 1 s (Methods percentiles):",
      {k: int(daach.stack(s, f).sum()) for k, f in (("ACh peak", lambda v: v["ACh"]["methods"][0]),
                                                  ("ACh dip", lambda v: v["ACh"]["methods"][1]),
                                                  ("DA peak", lambda v: v["DA"]["methods"][0]))})
print("Light-cue trials per mouse:", {m: len(d["events"]) for m, d in er4.items()})

**Result: not reproduced.** The light cue evokes an ACh peak, then a DA peak, then an ACh dip at
58 of 75 sites under both readings, more than the paper's 49 and above the tolerance (44.1 to 53.9).
The statement's qualitative part holds: the sequence is at a majority of sites overall and in each
mouse (12 of 17, 19 of 24, 27 of 34). With the code's percentiles the count is 55 or 52.

## fig4c-cue-correlation

> For conditioned light cues, ACh correlates with the trial-by-trial DA peak negatively at lagging
> times at 39 of 75 sites (202.28 +/- 128.4 ms) and positively at leading times at 26 of 75
> (-79.1 +/- 224.0 ms).

In [ ]:
res4 = {m: daach.event_correlation(d["rasters"]["DA"], d["rasters"]["ACh"], 1, IDX4, np.arange(0, 19)) for m, d in er4.items()}
dom4 = daach.stack(res4, lambda v: v["dominant"])
lat4 = daach.stack(res4, lambda v: v["lag"]) * 1000 / 18
sigDA4 = daach.stack(sig4[(9, 27)], lambda v: v["DA"]["methods"][0])
checks = [("negative", 39, 3.9), ("positive", 26, 2.6), ("mean_latency_negative_ms", 202.28, 20.6),
          ("mean_latency_positive_ms", -79.1, 43.9)]
for label, sel in (("(a) sites with a significant DA peak", sigDA4), ("(b) all sites", np.ones_like(sigDA4))):
    sm_ = corr_summary(dom4[sel], lat4[sel])
    print(label, {k: round(v, 1) if isinstance(v, float) else v for k, v in sm_.items()})
    print("   within tolerance:", {k: abs(sm_[k] - e) <= t for k, e, t in checks})
print(pd.DataFrame([{"mouse": m, **{k: (round(v, 1) if isinstance(v, float) else v) for k, v in
                     corr_summary(dom4[mice4 == m], lat4[mice4 == m]).items()}} for m in np.unique(mice4)]).to_string(index=False))

**Result: partially reproduced.** All 75 sites have a significant cue DA peak, so both readings
give the same values: 41 negative sites with a mean latency of 206.0 ms (the paper 39 and 202.28 ms)
and 34 positive sites with a mean latency of -83.3 ms (the paper 26 and -79.1 ms). Three values are
within tolerance; the positive count is above its range (23.4 to 28.6). ACh lags at 37 of the 41
negative sites and leads at 22 of the 34 positive ones.

## fig4h-extinction-correlation

> After cue-specific extinction, cue-evoked DA dips correlate negatively with lagging ACh at 41 of 74
> sites, mean lag 117.9 +/- 176.2 ms.

In [ ]:
rng = np.random.default_rng(402)
er4x = daach.event_rasters(ext, light_events, *W4, exclude_of=iti_exclusion)
sig4x = daach.triggered_significance(er4x, *W4, (9, 36), rng)
mice4x, coords4x = daach.site_table(ext)
assert np.allclose(coords4x, coords4)
dip4x = daach.stack(sig4x, lambda v: v["DA"]["methods"][1])
print("Light-cue trials per mouse:", {m: len(d["events"]) for m, d in er4x.items()})
print("Sites with a significant DA dip in 0 to 1.5 s:", int(dip4x.sum()), "of", len(dip4x),
      "; by mouse", pd.Series(dip4x).groupby(mice4x).sum().to_dict(),
      "; code percentiles:", int(daach.stack(sig4x, lambda v: v["DA"]["code"][1]).sum()))
for pol, label in ((-1, "DA trough (Methods, main)"), (1, "DA peak (code's polarity, variant)")):
    rx = {m: daach.event_correlation(d["rasters"]["DA"], d["rasters"]["ACh"], pol, IDX4, np.arange(0, 28)) for m, d in er4x.items()}
    d4x = daach.stack(rx, lambda v: v["dominant"])
    l4x = daach.stack(rx, lambda v: v["lag"]) * 1000 / 18
    if pol == -1:
        dom4x, lat4x = d4x, l4x
    for rl, sel in (("(a) significant-dip sites", dip4x), ("(b) all sites", np.ones_like(dip4x))):
        neg = sel & (d4x < 0)
        print(f"{label}, {rl}: {int(sel.sum())} sites, negative {int(neg.sum())} (lag > 0: {int((neg & (l4x > 0)).sum())}), "
              f"mean lag {l4x[neg].mean():.1f} ms (sd {l4x[neg].std(ddof=1):.1f}); within tolerance: count "
              f"{abs(neg.sum() - 41) <= 4.1}, lag {abs(l4x[neg].mean() - 117.9) <= 27.5}")
print(pd.DataFrame([{"mouse": m, "dip_sites": int(dip4x[mice4x == m].sum()),
                     "negative": int((dip4x & (dom4x < 0))[mice4x == m].sum()),
                     "mean_lag_neg": lat4x[(mice4x == m) & dip4x & (dom4x < 0)].mean()} for m in np.unique(mice4x)]).round(1).to_string(index=False))

**Result: reproduced.** Under reading (a), 38 of the 61 sites with a significant cue DA dip have a
dominant negative correlation with a mean lag of 118.4 ms (the paper 41 sites and 117.9 ms); both
are within tolerance. Over all 75 sites (reading b) 47 are negative with a mean lag of 114.7 ms. ACh
lags at 28 of the 38 negative sites, and in AD2 the mean lag of negative sites is -79.4 ms. The
code's polarity (the DA peak) gives 44 negative sites with a mean lag of 137.6 ms.

## fig4d-cue-in-out

> Conditioned-cue DA-ACh correlations are more negative at sites inside the global aDLS hotspot than
> outside.

In [ ]:
in4 = daach.inside(GRID, hs1, coords4)
print("AD1-AD3 sites inside the global hotspot:", int(in4.sum()), "by mouse", pd.Series(in4).groupby(mice4).sum().to_dict())
print("Sites with a significant DA peak:", daach.in_out(np.where(sigDA4, dom4, np.nan), in4))
print("Variant, all sites:", daach.in_out(dom4, in4))
interp4 = daach.Interpolator(GRID, coords4)
rng = np.random.default_rng(403)
hs4 = daach.hotspot(interp4, dom4, rng)
print(f"Variant, cue hotspot: mean {hs4['mean']:.3f}, centroid {hs4['centroid'].round(2)}, overlap",
      daach.overlap_test(hs1, interp1.inside, hs4, interp4.inside, GRID.shape, rng))

**Result: reproduced.** The 9 AD1 to AD3 sites inside the global hotspot have a median cue r of
-0.31 against 0.02 for the 66 outside (p = 0.023). A hotspot drawn from the cue correlations
themselves is negative, but its overlap with the global hotspot is not significant (147 voxels,
p = 0.175).

## fig4j-extinction-in-out

> After extinction, DA dip to ACh correlations are more negative at sites inside the global aDLS
> hotspot than outside.

In [ ]:
print("Sites with a significant DA dip:", daach.in_out(np.where(dip4x, dom4x, np.nan), in4))
print("Variant, all sites:", daach.in_out(dom4x, in4))
rng = np.random.default_rng(404)
hs4x = daach.hotspot(daach.Interpolator(GRID, coords4[dip4x]), dom4x[dip4x], rng)
print(f"Variant, extinction hotspot over dip sites: mean {hs4x['mean']:.3f}, centroid {hs4x['centroid'].round(2)}, overlap",
      daach.overlap_test(hs1, interp1.inside, hs4x, daach.Interpolator(GRID, coords4[dip4x]).inside, GRID.shape, rng))

**Result: reproduced.** Among sites with a significant DA dip, the 8 inside the global hotspot
have a median r of -0.35 against -0.21 for the 53 outside (p = 0.034); over all sites p = 0.018. The
extinction hotspot's own overlap with the global hotspot is not significant (128 voxels, p = 0.17).

## fig5d-initiation-responses

> At locomotion initiations, ACh rises and then falls at 131 of 174 sites; DA dips at 119 of 174
> sites, rises at 73, and both rises and dips at 32.

In [ ]:
W5 = (-18, 36)
IDX5 = np.arange(W5[0], W5[1] + 1)
SIG5 = (9, 36)  # rows for -0.5 to 1 s


def init_rasters(loco):
    out = {}
    for m, d in loco.items():
        samp = daach.sampleable_times(len(d["vel"]), *W5, valid=~np.isnan(d["vel"]))
        samp = np.setdiff1d(samp, d["init"])
        out[m] = {"cat": d["cat"], "events": d["init"], "sampleable": samp,
                  "rasters": {k: daach.raster(d["cat"][k], d["init"], *W5) for k in ("DA", "ACh")}}
    return out


def init_counts(sig, alpha="methods"):
    dp = daach.stack(sig, lambda v: v["DA"][alpha][0]); dn = daach.stack(sig, lambda v: v["DA"][alpha][1])
    ap = daach.stack(sig, lambda v: v["ACh"][alpha][0]); an = daach.stack(sig, lambda v: v["ACh"][alpha][1])
    ta = daach.stack(sig, lambda v: np.argmax(v["ACh"]["mean"][SIG5[0]:SIG5[1] + 1], 0))
    tn = daach.stack(sig, lambda v: np.argmin(v["ACh"]["mean"][SIG5[0]:SIG5[1] + 1], 0))
    return {"ACh rise then fall (a)": int((ap & an & (ta < tn)).sum()), "ACh rise (b)": int(ap.sum()),
            "DA dip": int(dn.sum()), "DA rise": int(dp.sum()), "DA both": int((dp & dn).sum()),
            "ACh dip": int(an.sum()), "ACh both, any order": int((ap & an).sum())}

loco = daach.locomotion_events(spont)
print("Initiations (bout >= 2.27 s) per mouse:", {m: len(d["init"]) for m, d in loco.items()},
      "; all clean onsets:", {m: len(d["init_all"]) for m, d in loco.items()})
er5 = init_rasters(loco)
sig5 = daach.triggered_significance(er5, *W5, SIG5, np.random.default_rng(501))
c5 = init_counts(sig5)
print("Counts (Methods percentiles):", c5)
exp5 = {"DA dip": 119, "DA rise": 73, "DA both": 32}
print("Within 10%:", {k: abs(c5[k] - v) <= 0.1 * v for k, v in exp5.items()},
      "; ACh (a)", abs(c5["ACh rise then fall (a)"] - 131) <= 13.1, "; ACh (b)", abs(c5["ACh rise (b)"] - 131) <= 13.1)
print("Variant, code percentiles (0.5/99.5):", init_counts(sig5, "code"))
mice5 = np.concatenate([[m] * d["cat"]["DA"].shape[1] for m, d in loco.items()])
per5 = pd.DataFrame([{"mouse": m, **init_counts({m: sig5[m]})} for m in sig5])
print(per5.to_string(index=False))
loco22 = daach.locomotion_events(spont, bout_s=2.2)
sig22 = daach.triggered_significance(init_rasters(loco22), *W5, SIG5, np.random.default_rng(502))
print("Variant, the code's bout of 2.2 s:", {m: len(d["init"]) for m, d in loco22.items()}, init_counts(sig22))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 3.2), sharex=True)
t = IDX5 / 18
for ax, ch in zip(axs, ("DA", "ACh")):
    mean = np.concatenate([sig5[m][ch]["mean"] for m in sig5], 1)
    dn = daach.stack(sig5, lambda v: v[ch]["methods"][1])
    dp = daach.stack(sig5, lambda v: v[ch]["methods"][0])
    ax.plot(t, mean[:, dn & ~dp].mean(1), label=f"dip only (n {int((dn & ~dp).sum())})")
    ax.plot(t, mean[:, dp & ~dn].mean(1), label=f"rise only (n {int((dp & ~dn).sum())})")
    ax.plot(t, mean[:, dp & dn].mean(1), label=f"both (n {int((dp & dn).sum())})")
    ax.axvline(0, color="k", lw=0.5)
    ax.set_title(f"{ch} at initiation"); ax.set_xlabel("s"); ax.legend(fontsize=7)
    print(ch, "plotted means range", np.nanmin(mean.mean(1)).round(4), "to", np.nanmax(mean.mean(1)).round(4))
axs[0].set_ylabel("dF/F")
plt.show()

**Result: partially reproduced.** At initiations, 129 sites have a significant DA dip and 77 a
DA rise (the paper 119 and 73, both within 10%), but 46 have both (the paper 32). The ACh count is
107 under reading (a), a rise followed by a dip, and 147 under reading (b), a rise; neither is
within 117.9 to 144.1. The code's percentiles give 114 dips, 61 rises and 26 sites with both. UG27
contributes 501 initiations, three to five times as many as each other mouse.

## fig5-latency-order

> Initiation-related signals occur in sequence; DA dips first (median 111.11 ms), then ACh peaks
> (222.22 ms) and DA peaks (227.78 ms), then ACh dips (611.11 ms).

In [ ]:
def latencies(sig, alpha="methods"):
    out = {}
    for ch in ("DA", "ACh"):
        mean = np.concatenate([sig[m][ch]["mean"] for m in sorted(sig)], 1)[SIG5[0]:SIG5[1] + 1]
        t = IDX5[SIG5[0]:SIG5[1] + 1] * 1000 / 18
        pk = daach.stack(sig, lambda v: v[ch][alpha][0]); dp = daach.stack(sig, lambda v: v[ch][alpha][1])
        out[f"{ch} peak"] = t[np.argmax(mean, 0)][pk]
        out[f"{ch} dip"] = t[np.argmin(mean, 0)][dp]
    return out

lat5 = latencies(sig5)
exp_l = {"DA dip": 111.11, "ACh peak": 222.22, "DA peak": 227.78, "ACh dip": 611.11}
rows = [{"feature": k, "sites": len(v), "median_ms": np.median(v), "IQR": np.percentile(v, [25, 75]).round(1),
         "paper": exp_l[k], "within_55.6ms": abs(np.median(v) - exp_l[k]) <= 55.6} for k, v in lat5.items()]
print(pd.DataFrame(rows).to_string(index=False))
m_ = {k: np.median(v) for k, v in lat5.items()}
print("Order DA dip < ACh peak, DA peak < ACh dip:", m_["DA dip"] < min(m_["ACh peak"], m_["DA peak"]) and
      max(m_["ACh peak"], m_["DA peak"]) < m_["ACh dip"])
print("By mouse (median ms):")
print(pd.DataFrame({m: {k: np.median(v) if len(v) else np.nan for k, v in latencies({m: sig5[m]}).items()} for m in sig5}).round(1).to_string())
print("Variant, code percentiles:", {k: round(float(np.median(v)), 1) for k, v in latencies(sig5, "code").items()})

**Result: reproduced.** Median latencies are 111.1 ms for DA dips, 277.8 ms for ACh peaks and for
DA peaks, and 666.7 ms for ACh dips: the paper's order, each within one sample of its value
(111.11, 222.22, 227.78 and 611.11 ms). The ACh peak and ACh dip medians are exactly one sample later
than the paper's, at the edge of the tolerance. Per mouse the order is less regular: the DA peak
median ranges from -388.9 to 527.8 ms, and UG31's DA dip median is 1000 ms.

## fig5e-initiation-topography

> Initiation-related DA dips are concentrated in the aDLS and DA peaks in dorsomedial and posterior
> striatum; ACh dips are localized to the aDLS.

In [ ]:
mean_da = np.concatenate([sig5[m]["DA"]["mean"] for m in sorted(sig5)], 1)[SIG5[0]:SIG5[1] + 1]
mx, mn = mean_da.max(0), mean_da.min(0)
dom_da = np.where(np.abs(mn) > np.abs(mx), mn, mx)
ach_dip = daach.stack(sig5, lambda v: v["ACh"]["methods"][1])
coords5 = coords1
rows = []
for label, a, b in (("negative vs positive dominant DA", dom_da < 0, dom_da > 0), ("ACh dip vs none", ach_dip, ~ach_dip)):
    for d, name in ((0, "AP"), (1, "ML"), (2, "DV")):
        p = stats.mannwhitneyu(coords5[a, d], coords5[b, d], alternative="greater").pvalue
        rows.append({"comparison": label, "axis": name, "n": f"{int(a.sum())} vs {int(b.sum())}",
                     "median_a": np.median(coords5[a, d]), "median_b": np.median(coords5[b, d]), "one_sided_p_greater": p})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("Inside the spontaneous hotspot: DA negative", int((in1 & (dom_da < 0)).sum()), "of", int(in1.sum()),
      "; ACh dip", int((in1 & ach_dip).sum()), "of", int(in1.sum()))

**Result: not reproduced.** Sites whose dominant DA response is a dip are more anterior than sites
whose dominant response is a rise (median AP 0.36 against -0.48 mm, p < 0.0001), but more medial (ML
1.88 against 2.30 mm; p = 0.996 for lateral) and more ventral, so part (i) fails on ML. Sites with a
significant ACh dip (148 of 174) are not more anterior or lateral than the 26 without (p = 0.33 and
0.25). All 28 sites inside the correlation hotspot have a dominant DA dip and a significant ACh dip.

## fig6a-initiation-correlation

> At sites with initiation-related DA dips (119 sites), trial-by-trial DA dip magnitudes correlate
> negatively with ACh with DA leading (mean latency 193.80 +/- 177.68 ms).

In [ ]:
res6 = {m: daach.event_correlation(d["rasters"]["DA"], d["rasters"]["ACh"], -1, IDX5, np.arange(-9, 19)) for m, d in er5.items()}
dom6 = daach.stack(res6, lambda v: v["dominant"])
lat6 = daach.stack(res6, lambda v: v["lag"]) * 1000 / 18
dip5 = daach.stack(sig5, lambda v: v["DA"]["methods"][1])
neg = dip5 & (dom6 < 0)
print(f"Significant-dip sites {int(dip5.sum())}; dominant negative {int(neg.sum())} ({neg.sum() / dip5.sum():.2f}); "
      f"negative with lag > 0: {int((neg & (lat6 > 0)).sum())}")
print(f"(a) mean latency over negative sites {lat6[neg].mean():.1f} ms (sd {lat6[neg].std(ddof=1):.1f}); "
      f"(b) over all dip sites {lat6[dip5].mean():.1f} ms (sd {lat6[dip5].std(ddof=1):.1f}); paper 193.80 +/- 177.68, tolerance 16.3")
print(pd.DataFrame([{"mouse": m, "dip_sites": int(dip5[mice5 == m].sum()), "negative": int(neg[mice5 == m].sum()),
                     "mean_latency_neg": lat6[neg & (mice5 == m)].mean()} for m in np.unique(mice5)]).round(1).to_string(index=False))

**Result: reproduced.** Of the 129 sites with a significant DA dip, 99 (77%) have a dominant
negative correlation, 89 of them with ACh lagging, with a mean latency of 182.4 ms (reading a; the
paper 193.80 ms, tolerance 177.5 to 210.1). Over all dip sites (reading b) the mean is 99.1 ms. Most
dip sites are negative in every mouse.

## fig6c-initiation-in-out

> Initiation DA dip to ACh correlations are concentrated in the aDLS and more negative inside the
> global hotspot than outside.

In [ ]:
print(daach.in_out(np.where(dip5, dom6, np.nan), in1))
print("Variant, all sites:", daach.in_out(dom6, in1))

**Result: reproduced.** Among dip sites the initiation correlation is more negative inside the
hotspot (median -0.36, 28 sites) than outside (-0.21, 101 sites), p = 6.1e-08.

## fig6g-invigoration-in-out

> At locomotion invigorations, GLM-estimated DA peak magnitudes correlate negatively with ACh at
> lagging times, and these correlations are more negative inside the global hotspot (131 sites with
> invigoration DA peaks).

In [ ]:
t0 = time.time()
inv = daach.invigoration_analysis(loco)
print(f"GLM fits in {time.time() - t0:.0f} s; invigorations per mouse:", {m: v["n_invig"] for m, v in inv.items()})
sig_inv = daach.stack(inv, lambda v: v["sig_da_peak"])
res6g = {m: daach.event_correlation(v["rasters"]["DA"], v["rasters"]["ACh"], 1, IDX5, np.arange(-9, 19)) for m, v in inv.items()}
dom6g = daach.stack(res6g, lambda v: v["dominant"])
lat6g = daach.stack(res6g, lambda v: v["lag"]) * 1000 / 18
neg = sig_inv & (dom6g < 0)
print(f"Sites with an invigoration DA peak: {int(sig_inv.sum())} (paper 131); dominant negative {int(neg.sum())}, "
      f"lag > 0 among them {int((neg & (lat6g > 0)).sum())}, mean lag {lat6g[neg].mean():.1f} ms")
print("In vs out among selected sites:", daach.in_out(np.where(sig_inv, dom6g, np.nan), in1))
print("Variant, all sites:", daach.in_out(dom6g, in1))
print(pd.DataFrame([{"mouse": m, "selected": int(sig_inv[mice5 == m].sum()), "negative": int(neg[mice5 == m].sum())}
                    for m in np.unique(mice5)]).to_string(index=False))

**Result: reproduced.** The GLM finds an invigoration DA peak at 160 sites (the paper 131). Of
these, 105 have a dominant negative correlation, 97 of them with ACh lagging (mean lag 179.4 ms), and
the correlations are more negative inside the hotspot (median -0.34, 28 sites) than outside (-0.21,
132 sites), p = 4.5e-05.

## fig7-da-release

> Optogenetic stimulation of midbrain DA neurons evokes significant DA release at 179 of 180 sites.

In [ ]:
W7 = (-9, 45)
SIG7 = (7, 45)  # rows for -2 to 36 samples


def iti_stims(s):
    st = daach.event_index(s, s["events"]["Optogenetic Stimulation"][0])
    rew = daach.event_index(s, s["events"]["Reward"][0])
    return np.array([i for i in st if not (np.abs(rew - i) <= 18).any()], int)


def stim_exclusion(s):
    rew, lick = daach.first_licks(s)
    st = daach.event_index(s, s["events"]["Optogenetic Stimulation"][0])
    return ([(i, i + int(5 * daach.SR) + 1) for i in np.concatenate([rew, lick])] + [(i, i + 19) for i in st])

er7 = daach.event_rasters(ads, iti_stims, *W7, exclude_of=stim_exclusion)
sig7 = daach.triggered_significance(er7, *W7, SIG7, np.random.default_rng(701))
print("ITI stimulation trains per mouse:", {m: len(d["events"]) for m, d in er7.items()},
      "; all trains:", {m: int(sum(len(s["events"]["Optogenetic Stimulation"][0]) for s in v)) for m, v in daach.by_mouse(ads).items()})
rows = []
for m in sig7:
    rows.append({"mouse": m, "sites": len(sig7[m]["DA"]["methods"][0]), "DA_peak": int(sig7[m]["DA"]["methods"][0].sum()),
                 "DA_peak_code": int(sig7[m]["DA"]["code"][0].sum()), "ACh_dip": int(sig7[m]["ACh"]["methods"][1].sum()),
                 "ACh_dip_code": int(sig7[m]["ACh"]["code"][1].sum())})
t7 = pd.DataFrame(rows)
print(t7.to_string(index=False))
print("Totals:", t7.drop(columns="mouse").sum().to_dict())

**Result: reproduced.** Stimulation trains in the ITI (20 per mouse, 15 for ADS6) evoke a
significant DA peak at 179 of 180 sites (178 with the code's percentiles), as in the paper.

## fig7e-ach-suppression

> DA stimulation evokes significant ACh suppression at a subset of sites (116 of 180, 64%).

In [ ]:
n_dip = int(t7.ACh_dip.sum())
print(f"Sites with a significant ACh dip: {n_dip} of {int(t7.sites.sum())} ({n_dip / t7.sites.sum():.2f}); "
      f"code percentiles {int(t7.ACh_dip_code.sum())}; paper 116 (0.64), tolerance 11.6")
fig, ax = plt.subplots(figsize=(4.5, 3))
for ch, c in (("DA", "tab:red"), ("ACh", "tab:green")):
    mean = np.concatenate([sig7[m][ch]["mean"] for m in sorted(sig7)], 1)
    ax.plot(np.arange(W7[0], W7[1] + 1) / 18, mean.mean(1), color=c, label=ch)
    print(ch, "mean over sites, range", mean.mean(1).min().round(4), "to", mean.mean(1).max().round(4))
ax.axvline(0, color="k", lw=0.5); ax.set_xlabel("s from stimulation onset"); ax.set_ylabel("dF/F"); ax.legend()
plt.show()

**Result: partially reproduced.** A significant ACh dip follows stimulation at 147 of 180 sites
(82%), against 116 (64%) in the paper; with the code's percentiles 128 (71%). The 33 sites without a
dip are concentrated in ADS19 (17 of its 27 sites) and ADS16 (9 of 37).

## ed9-cohort-hotspot

> The stimulation cohort replicates the aDLS anti-correlation hotspot in spontaneous release outside
> stimulation periods.

In [ ]:
def ads_mask(mode):
    def f(s):
        n = len(s["DA"])
        rew, lick = daach.first_licks(s)
        st = daach.event_index(s, s["events"]["Optogenetic Stimulation"][0])
        a, b, c = daach.time_since(n, rew) < 6, daach.time_since(n, lick) < 6, daach.time_since(n, st) < 3
        return (a & b & c) if mode == "and" else (a | b | c)
    return f

xc9 = daach.spontaneous_xcorr(ads, mask=ads_mask("and"), concatenate=True)
dom9, lag9 = xc9["dominant"], xc9["lag"]
print("Cohort counts (code's AND exclusion):", polarity_counts(dom9, lag9))
in9 = daach.inside(GRID, hs1, coords_s)
print("Sites inside the original hotspot:", int(in9.sum()), "by mouse", pd.Series(in9).groupby(mice_s).sum().to_dict())
print("(i) in vs out of the original hotspot:", daach.in_out(dom9, in9))
interp9 = daach.Interpolator(GRID, coords_s)
rng = np.random.default_rng(901)
hs9 = daach.hotspot(interp9, dom9, rng)
ov9 = daach.overlap_test(hs1, interp1.inside, hs9, interp9.inside, GRID.shape, rng)
print(f"(ii) cohort hotspot: mean {hs9['mean']:.3f}, voxels {ov9['n_b']}, centroid {hs9['centroid'].round(2)}; overlap", ov9)
xc9o = daach.spontaneous_xcorr(ads, mask=ads_mask("or"), concatenate=True)
print("Variant, OR exclusion:", polarity_counts(xc9o["dominant"], xc9o["lag"]), daach.in_out(xc9o["dominant"], in9))
hs9o = daach.hotspot(interp9, xc9o["dominant"], rng)
print("Variant, OR exclusion hotspot overlap:", daach.overlap_test(hs1, interp1.inside, hs9o, interp9.inside, GRID.shape, rng))

**Result: partially reproduced.** In the stimulation cohort, the 9 sites inside the original
hotspot have more negative dominant correlations (median -0.42) than the 171 outside (-0.30),
p = 0.049; the OR exclusion gives p = 0.041. The cohort's own hotspot is negative (mean -0.44, 1536
voxels, centroid AP 0.71, ML 2.00, DV -3.45 mm) and shares 564 voxels with the original against a
null median of 65, but the overlap test gives p = 0.085. 167 of the 180 sites have a dominant
negative correlation.

## fig7f-ach-dip-in-out

> Stimulation-evoked ACh dips are larger inside the cohort's correlation hotspot than outside, and
> their own hotspot overlaps it.

In [ ]:
amin = np.concatenate([np.nanmean(np.nanmin(er7[m]["rasters"]["ACh"][SIG7[0]:SIG7[1] + 1], 0), 1) for m in sorted(er7)])
in9c = daach.inside(GRID, hs9, coords_s)
print("Sites inside the cohort hotspot:", int(in9c.sum()))
print("ACh minima, in vs out of the cohort hotspot:", daach.in_out(amin, in9c))
print("Variant, in vs out of the original hotspot:", daach.in_out(amin, in9))
rng = np.random.default_rng(902)
hs7 = daach.hotspot(interp9, amin, rng, how="min")
print(f"Variant, ACh-minimum hotspot: mean {hs7['mean']:.4f}, centroid {hs7['centroid'].round(2)}; overlap with cohort hotspot",
      daach.overlap_test(hs9, interp9.inside, hs7, interp9.inside, GRID.shape, rng))
print(pd.DataFrame([{"mouse": m, **daach.in_out(amin[mice_s == m], in9c[mice_s == m])} for m in np.unique(mice_s)]).round(4).to_string(index=False))

**Result: reproduced.** ACh minima after stimulation are deeper at the 17 sites inside the cohort
hotspot (median -0.0179 dF/F) than at the 163 outside (-0.0149), p = 0.0005. Against the original
hotspot (9 sites inside) the difference has the same sign but p = 0.094, and the ACh-minimum
hotspot's overlap with the cohort hotspot gives p = 0.065. Within each of the six mice the inside
sites have lower minima, significantly so in one (ADS19).

## fig8b-quinpirole-region

> Quinpirole suppresses evoked ACh release in DLS and NAcC, more strongly in DLS; region F(1,40) =
> 105.9, interaction F(4,40) = 3.014, stimulus F(4,40) = 17.80.

The slice data are not in the dandiset. The paper's source data for Figure 8 are on Zenodo
(doi:10.5281/zenodo.18202873) and are read from there.

In [ ]:
FIG8 = "https://zenodo.org/records/18202873/files/Bouabid_Vu%20%282026%29_Source_Data_Fig8.xlsx?download=1"
fig8 = pd.read_excel(FIG8, sheet_name=None)
print("Sheets:", list(fig8))


def anova(sheet, factor):
    d = fig8[sheet].rename(columns={factor: "g"})
    fit = smf.ols("Peak_ACh ~ C(g) * C(Stim)", data=d).fit()
    tab = sm.stats.anova_lm(fit, typ=2)
    print(sheet, "| rows", len(d), "| slices per group", d.groupby("g").Slice_ID.nunique().to_dict(),
          "| mouse ids per group", d.groupby("g").Mouse_ID.nunique().to_dict(), "| group means", d.groupby("g").Peak_ACh.mean().round(1).to_dict())
    print(tab.round(4).to_string())
    return tab

t8b = anova("Figure_8b", "Brain_Region")
print(fig8["Figure_8b"].groupby(["Brain_Region", "Slice_ID"]).Stim.apply(list).to_string())

**Result: reproduced.** The two-way ANOVA on the source data gives region F = 105.87, stimulus
F = 17.80 and interaction F = 3.014 (the paper 105.9, 17.80 and 3.014), with DLS lower (25.1 against
68.1). The DLS group holds 6 slice labels from 4 mouse labels, one of them (slice_B_4) with only
stimulus 5, where the legend states 5 slices from 5 mice.

## fig8e-cocaine-region

> Cocaine reduces evoked ACh release more in DLS than in NAcC; interaction F(4,55) = 10.75, stimulus
> F(4,55) = 36.37, region F(1,55) = 199.9.

In [ ]:
t8e = anova("Figure_8e", "Brain_Region")

**Result: partially reproduced.** Region F = 199.86 and interaction F = 10.75 match the paper
(199.9 and 10.75), with DLS lower (32.8 against 74.1), but the stimulus F is 47.68 against the
paper's 36.37.

## fig8fg-d2-block

> The D2 antagonist attenuates the cocaine-induced suppression in both regions; block effect
> F(1,55) = 231.9 in DLS and F(1,40) = 32.57 in NAcC.

In [ ]:
t8f = anova("Figure_8f", "Condition")
t8g = anova("Figure_8g", "Condition")

**Result: reproduced.** The D2R block effect is F = 231.87 in DLS and 32.57 in NAcC (the paper
231.9 and 32.57), and the antagonist raises ACh in both (DLS 32.8 to 78.6, NAcC 74.1 to 107.9). The
DLS stimulus F is 45.04, where the legend states 34.46.

## fig8c-cocaine-da-similar (blocked)

> Cocaine elevates evoked DA release to a similar degree in DLS and NAcC (cocaine by region
> interaction F(8,36) = 0.20).

In [ ]:
for sh in ("Figure_8c_LP", "Figure_8c_RP"):
    d = fig8[sh]
    print(sh, "| mice", d.Mouse_ID.unique().tolist(), "| conditions", d.Condition.unique().tolist(),
          "| time points", d.Timepoints.nunique(), d.Timepoints.min(), "to", d.Timepoints.max())

The sheets hold mean DA time courses (24 time points) per mouse and condition, three mice per
region. The legend's ANOVA has 8 degrees of freedom for cocaine and 36 residual, which fits neither
a two-level cocaine factor nor 24 time points, so the test cannot be repeated.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig1e-dominant-polarity | reproduced |
| fig1f-adls-hotspot | partially reproduced (centroid 0.02 mm medial of the median site) |
| fig1h-in-vs-out | reproduced |
| fig1f-positive-posterior | reproduced |
| ed2-hotspot-per-mouse | reproduced |
| ed1-controls-no-organization | not reproduced (control correlations as strong as sensor ones; a hotspot is found) |
| fig2a-directionality | reproduced |
| fig2-magnitude-correlations | partially reproduced |
| fig2d-magnitude-in-out | partially reproduced |
| fig2b-pairing-overlaps-hotspot | reproduced |
| ed3e-transient-rates-in-hotspot | reproduced |
| fig3b-reward-correlation | partially reproduced |
| fig3d-reward-in-out | reproduced |
| fig3e-reward-hotspot-overlap | reproduced |
| fig4-cue-sequence | not reproduced (58 of 75 sites, the paper 49) |
| fig4c-cue-correlation | partially reproduced |
| fig4h-extinction-correlation | reproduced |
| fig4d-cue-in-out | reproduced |
| fig4j-extinction-in-out | reproduced |
| fig5d-initiation-responses | partially reproduced |
| fig5-latency-order | reproduced |
| fig5e-initiation-topography | not reproduced |
| fig6a-initiation-correlation | reproduced |
| fig6c-initiation-in-out | reproduced |
| fig6g-invigoration-in-out | reproduced |
| fig7-da-release | reproduced |
| fig7e-ach-suppression | partially reproduced |
| ed9-cohort-hotspot | partially reproduced |
| fig7f-ach-dip-in-out | reproduced |
| fig8b-quinpirole-region (Zenodo data) | reproduced |
| fig8e-cocaine-region (Zenodo data) | partially reproduced |
| fig8fg-d2-block (Zenodo data) | reproduced |
| fig8c-cocaine-da-similar (Zenodo data) | blocked, methods underspecified |

**How the analysis differs from the paper's.** The hotspot procedure is a simplified port of the
paper's MATLAB pipeline: linear in place of natural-neighbor interpolation, 100 um in place of 50 um
voxels, the convex hull of the sites in place of the Allen striatum mask, and 200 in place of 10000
shuffles and random volumes. Bootstrap nulls use 1000 samples in place of 10000. MATLAB's `lowpass`
is replaced by a zero-phase Butterworth filter. Pavlovian phases are inferred from the files'
events. The control analysis uses the 15 uncued reward sessions of AD4 to AD6. Pairs of transients
are matched by the Methods' rule (the first partner), a simplification of the code's matching.

**What was found about the dandiset.** The photometry series state a rate of 9 Hz; the timestamps,
the Methods and the code give 18 Hz. The files do not say which experiment, cohort or Pavlovian phase
a session belongs to; this was inferred from subject names, indicators and events. Tone cue events
are absent from the Pavlovian files. One reward session of UG27 (240313r) lacks the artifact-removed
green series. The site counts match the paper (174, 75 and 180), and the data support every
spontaneous, reward and locomotion analysis tested. The excitation source descriptions are copied
from another setup (405 nm isosbestic light at 100 Hz), the mutant ACh sensor is named
ACh4.3mut where the paper names ACh3.0-mut, and the session description's reward volume and
intervals differ from the Methods. The paper's Figure 8 source data are on
Zenodo, not in the dandiset.

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min")